In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [12]:
!pip install dagshub mlflow xgboost imbalanced-learn category_encoders --quiet

import numpy as np
import pandas as pd
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(
    repo_owner="lmars23",
    repo_name="ml_assignment_2",
    mlflow=True
)

mlflow.set_tracking_uri("https://dagshub.com/lmars23/ml_assignment_2.mlflow")

model_name = "model_ieee_fraud_xgboost"
model_name = "model_ieee_fraud_xgboost_permutation_selected"

model = mlflow.sklearn.load_model(
    model_uri=f"models:/{model_name}/latest"
)

print("Loaded model:", model_name)

data_path = "/kaggle/input/competitions/ieee-fraud-detection"

test_transaction = pd.read_csv(f"{data_path}/test_transaction.csv")
test_identity = pd.read_csv(f"{data_path}/test_identity.csv")
sample_submission = pd.read_csv(f"{data_path}/sample_submission.csv")

test_identity.columns = test_identity.columns.str.replace("-", "_", regex=False)

test = test_transaction.merge(test_identity, on="TransactionID", how="left")

X_test = test.drop("TransactionID", axis=1)

print("test_transaction shape:", test_transaction.shape)
print("test_identity shape:", test_identity.shape)
print("merged test shape:", test.shape)
print("X_test shape:", X_test.shape)

preds = model.predict_proba(X_test)[:, 1]

submission = sample_submission.copy()
submission["isFraud"] = preds

submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")
print(submission.head())
print(submission["isFraud"].describe())

Initialized MLflow to track repo "lmars23/ml_assignment_2"

Repository lmars23/ml_assignment_2 initialized!

Loaded model: model_ieee_fraud_xgboost_permutation_selected
test_transaction shape: (506691, 393)
test_identity shape: (141907, 41)
merged test shape: (506691, 433)
X_test shape: (506691, 432)
id_02 exists: True
Saved submission.csv
   TransactionID   isFraud
0        3663549  0.022814
1        3663550  0.027251
2        3663551  0.051799
3        3663552  0.062455
4        3663553  0.023611
count    506691.000000
mean          0.166728
std           0.202615
min           0.000366
25%           0.036257
50%           0.089360
75%           0.209033
max           0.999668
Name: isFraud, dtype: float64
